# ME2 - VCM on Raspberry Pi 5 — Personalized Wake and Intent Retraining

This experiment uses the current deduplicated human recording manifest. It compares two independently scratch-initialized two-class wake detectors:

1. **Broad negatives:** real wake takes as positives; Option B command speech, silence and background noise as non-wake.
2. **Personal hard negatives:** the same wake takes and broad negatives, plus the speaker's own non-wake commands and near-miss speech.

Training on wake recordings alone cannot teach a wake/non-wake decision; a binary detector needs negative examples. The intent candidate is scratch-trained on the Option B training partition plus a small, controlled fraction of exact-mapped personal command recordings. The canonical Option B INT8 model is the intent baseline. All splits are made after exact PCM deduplication. The Option B held-out test data are not loaded until the final cell, after all models and thresholds are frozen.

**Important limits:** the audit cell reports the current speaker IDs. Personal data are split by label and clip after exact PCM deduplication, not by speaker, so the same people may occur in train, validation, and test. These results do not establish unseen-speaker accuracy. Several fixed intent classes have very few unique personal takes. The Option B test split has been used in prior project experiments. No Pi performance claim is made by these notebook results.

Normal execution trains from scratch and evaluates once. Set `VCM_EXISTING_RUN_DIR` before executing to reconstruct the visible report from an already completed run without retraining or reopening the held-out Option B test.

In [1]:
from pathlib import Path
import os, sys, json, time
from datetime import datetime
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'tinyvcm_antigrav').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from its ME2 project folder so tinyvcm_antigrav can be found.')
WORKSPACE_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from tinyvcm_antigrav.personalized_retrain import (
    prepare_experiment, run_wake_pair, train_intent_candidate_from_data,
    _export_intent_candidate, evaluate_all,
    load_personal_records, load_optionb_train_val,
)
from tinyvcm_antigrav.recording_labels import map_recorded_intent

REUSE_RUN_DIR = Path(os.environ['VCM_EXISTING_RUN_DIR']).resolve() if os.environ.get('VCM_EXISTING_RUN_DIR') else None
RUN_DIR = REUSE_RUN_DIR or (PROJECT_ROOT / 'runs' / f"personalized-vcm-{datetime.now().strftime('%Y%m%d-%H%M%S')}")
print('Project:', PROJECT_ROOT)
print('Existing completed run for report reconstruction:' if REUSE_RUN_DIR else 'New scratch training output:', RUN_DIR)
print('Shared Python:', sys.executable)


Project: C:\Users\danda\Desktop\MEng AI Notebooks\AI 222 231\AI 231\Dumalaog_ME2 - Tiny Voice Command Model
Existing completed run for report reconstruction: C:\Users\danda\Desktop\MEng AI Notebooks\AI 222 231\AI 231\Dumalaog_ME2 - Tiny Voice Command Model\runs\personalized-vcm-retrain-20260928-190911
Shared Python: C:\Users\danda\Desktop\MEng AI Notebooks\AI 222 231\.venv\Scripts\python.exe


## 1. Audit, deduplicate, split, and prepare training data

This cell validates the local manifest and WAV headers, removes exact duplicate PCM before splitting, writes a split manifest under the new run, and reads only the Option B training and validation feature arrays. It deliberately does not open the Option B test features.

In [2]:
if REUSE_RUN_DIR:
    human, _ = load_personal_records(PROJECT_ROOT)
    audit = json.loads((RUN_DIR / 'dataset_audit.json').read_text(encoding='utf-8'))
    DATA = {**load_optionb_train_val(PROJECT_ROOT), 'project_root': PROJECT_ROOT,
            'human': human, 'human_audit': audit, 'output_dir': RUN_DIR}
    print('Reconstructed the deterministic personal data split. Train/validation features only; test is not opened.')
else:
    DATA = prepare_experiment(RUN_DIR, PROJECT_ROOT)
audit = DATA['human_audit']
display(pd.DataFrame([
    {'saved files': audit['raw_manifest_rows'], 'unique waveforms': audit['unique_pcm_groups'],
     'duplicate rows removed': audit['exact_duplicate_rows_collapsed'],
     'real speakers': ', '.join(audit['speakers']), 'cross-label PCM collisions': len(audit['cross_label_hash_collisions'])}
]))
display(pd.DataFrame([
    {'label': label, 'unique takes': count,
     'train': sum(i['label']==label and i['split']=='train' for i in DATA['human']),
     'validation': sum(i['label']==label and i['split']=='validation' for i in DATA['human']),
     'test': sum(i['label']==label and i['split']=='test' for i in DATA['human']),
     'mapped intent': map_recorded_intent(label) or 'wake / not mapped'}
    for label, count in audit['unique_label_groups'].items()
]))
print('Option B cache manifest SHA-256:', DATA['manifest_sha256'].item())
print('Personal manifest SHA-256:', audit['manifest_sha256'])


Reconstructed the deterministic personal data split. Train/validation features only; test is not opened.


,saved files,unique waveforms,duplicate rows removed,real speakers,cross-label PCM collisions
0,524,292,232,"person-01, person-02, person-03",0


,label,unique takes,train,validation,test,mapped intent
0,ALARM_6_00AM,9,6,1,2,ALARM_6_00AM
1,ALARM_8_00AM,10,7,1,2,ALARM_8_00AM
2,LIGHT_ON,7,4,1,2,LIGHT_ON
3,PAUSE,12,8,1,3,PAUSE
4,PLAY_MUSIC,11,7,1,3,PLAY_MUSIC
5,STOP,10,7,1,2,STOP
6,_background_noise_,9,6,1,2,wake / not mapped
7,_silence_,5,3,1,1,wake / not mapped
8,_unknown_,4,2,1,1,wake / not mapped
9,alarm_set,11,7,1,3,wake / not mapped


Option B cache manifest SHA-256: 3a7cb34b708414df055bb16a15ed2c9f7d850f450c88f02602079e06e961ab23
Personal manifest SHA-256: 0f58ee1cfe128fa08b24aa5b9a1b0eecf39687240b0dc99b8b284d6e4a79707e


## 2. Train the wake-model comparison

Each variant sees the same content-deduplicated, clip-held-out wake recordings, Option B command pool, and training schedule. The personal split is not speaker-disjoint. The second variant adds only training-split personal command/near-miss recordings. Validation selects each INT8 threshold under a zero-validation-false-accept constraint. The final test clips remain unopened.

In [3]:
if REUSE_RUN_DIR:
    WAKE_RUNS = {}
    for name, folder in (('broad', 'wake_broad_negatives'), ('hard', 'wake_with_personal_intents')):
        model_dir = RUN_DIR / folder / 'models'
        summary = json.loads((model_dir / 'export_summary.json').read_text(encoding='utf-8'))
        WAKE_RUNS[name] = {
            'summary': summary,
            'val_threshold': summary['validation_threshold_int8'],
            'fp32_path': model_dir / 'wake_personalized_fp32.onnx',
            'int8_path': model_dir / 'wake_personalized_int8.onnx',
        }
        history = json.loads((RUN_DIR / folder / 'history.json').read_text(encoding='utf-8'))['history']
        print(f"Loaded {name} wake model trained for {len(history)} epochs; selected epoch {summary['best_epoch']}.")
        display(pd.DataFrame(history))
else:
    WAKE_RUNS = run_wake_pair(DATA, epochs=30)
display(pd.DataFrame([
    {'variant': name, 'INT8 bytes': run['summary']['int8_size_bytes'],
     'validation threshold': run['val_threshold'],
     'FP32 validation wake recall': run['summary']['fp32_validation_threshold_metrics']['wake_recall'],
     'validation wake recall': run['summary']['int8_validation_threshold_metrics']['wake_recall'],
     'validation wake support': run['summary']['int8_validation_threshold_metrics']['wake_support'],
     'validation non-wake support': run['summary']['int8_validation_threshold_metrics']['nonwake_support'],
     'INT8 SHA-256': run['summary']['int8_sha256']}
    for name, run in WAKE_RUNS.items()
]))


Loaded broad wake model trained for 30 epochs; selected epoch 11.


,epoch,train_loss,train_accuracy,val_zero_FA_wake_recall,val_threshold,val_wake_support,val_nonwake_clip_support,epoch_seconds
0,1,0.246994,0.940208,0.000000,1.000000,7,1848,3.582536
1,2,0.063597,0.987907,0.857143,0.628435,7,1848,2.208028
2,3,0.038336,0.991266,0.714286,0.981270,7,1848,2.189295
3,4,0.019453,0.996641,1.000000,0.866221,7,1848,2.233131
4,5,0.010421,0.998656,0.857143,0.990643,7,1848,2.191145
5,6,0.011258,0.997985,0.571429,0.937399,7,1848,2.197902
6,7,0.013062,0.997313,0.714286,0.945465,7,1848,2.197446
7,8,0.013765,0.996641,0.714286,0.999930,7,1848,2.202774
8,9,0.007582,0.999328,0.714286,0.999913,7,1848,2.180747
9,10,0.015203,0.996977,0.857143,0.099206,7,1848,2.214362


Loaded hard wake model trained for 30 epochs; selected epoch 10.


,epoch,train_loss,train_accuracy,val_zero_FA_wake_recall,val_threshold,val_wake_support,val_nonwake_clip_support,epoch_seconds
0,1,0.296949,0.907141,0.142857,0.899688,7,1848,2.480897
1,2,0.150684,0.950368,0.571429,0.556299,7,1848,2.294472
2,3,0.089758,0.973743,0.857143,0.837336,7,1848,2.324902
3,4,0.059336,0.982709,0.857143,0.961535,7,1848,2.333005
4,5,0.037536,0.990714,0.857143,0.925966,7,1848,2.305899
5,6,0.030078,0.991995,1.000000,0.929365,7,1848,2.291097
6,7,0.022691,0.994557,1.000000,0.975996,7,1848,2.278454
7,8,0.020900,0.995517,1.000000,0.858171,7,1848,2.315360
8,9,0.012797,0.997759,1.000000,0.903981,7,1848,2.335310
9,10,0.013111,0.997438,1.000000,0.998662,7,1848,2.325175


,variant,INT8 bytes,validation threshold,FP32 validation wake recall,validation wake recall,validation wake support,validation non-wake support,INT8 SHA-256
0,broad,37425,0.999867,1.0,1.0,7,1848,35a400533826236a597fb75569ff60c777add5db10dce8...
1,hard,37425,0.998543,1.0,1.0,7,1848,f7d1c3f7826bc16426eac02c10c05e077502f28bbc764c...


## 3. Train the personalized intent candidate from scratch

This retains the 31 fixed Option B output classes. Only exact-mapped personal labels are added; unsupported phrases such as five-minute timers, thermostat direction, and the recorded seven-AM alarm are excluded instead of being mislabeled. The per-batch personal fraction is intentionally small. Checkpoint selection uses validation only.

In [4]:
if REUSE_RUN_DIR:
    INTENT_DIR = RUN_DIR / 'intent_personalized'
    INTENT_RUN = {'output_dir': INTENT_DIR,
                  'run_info': json.loads((INTENT_DIR / 'run_info.json').read_text(encoding='utf-8'))}
    INTENT_EXPORT = json.loads((INTENT_DIR / 'models' / 'export_summary.json').read_text(encoding='utf-8'))
    history = json.loads((INTENT_DIR / 'history.json').read_text(encoding='utf-8'))['history']
    print(f"Loaded scratch-trained personalized intent model; selected epoch {INTENT_RUN['run_info']['best_epoch']}.")
    display(pd.DataFrame(history))
else:
    INTENT_RUN = train_intent_candidate_from_data(DATA, epochs=35)
    INTENT_EXPORT = _export_intent_candidate(INTENT_RUN, DATA)
display(pd.DataFrame([{
    'epochs': INTENT_RUN['run_info']['epochs'], 'best epoch': INTENT_RUN['run_info']['best_epoch'],
    'personal train clips': INTENT_RUN['run_info']['personal_unique_train_clips'],
    'parameter count': INTENT_RUN['run_info']['parameter_count'],
    'FP32 bytes': INTENT_EXPORT['fp32_size_bytes'], 'INT8 bytes': INTENT_EXPORT['int8_size_bytes'],
    'INT8 SHA-256': INTENT_EXPORT['int8_sha256']
}]))


Loaded scratch-trained personalized intent model; selected epoch 31.


,epoch,train_loss,train_accuracy,val_optionb_macro_f1,val_personal_macro_f1,checkpoint_score,epoch_seconds
0,1,2.838331,0.140694,0.157923,0.026667,0.131672,10.693975
1,2,2.244565,0.271042,0.287059,0.016667,0.232981,10.169088
2,3,1.887483,0.400556,0.400671,0.155556,0.351648,10.246706
3,4,1.651797,0.491736,0.496648,0.333333,0.463985,10.232574
4,5,1.467188,0.579792,0.588858,0.472698,0.565626,10.377866
5,6,1.305197,0.648333,0.689496,0.369206,0.625438,10.472956
6,7,1.186524,0.700972,0.745477,0.528889,0.702159,10.236756
7,8,1.080338,0.743194,0.808019,0.626667,0.771749,10.153116
8,9,1.008932,0.765556,0.829349,0.538889,0.771257,10.199638
9,10,0.956666,0.787500,0.833254,0.521587,0.770921,10.347389


,epochs,best epoch,personal train clips,parameter count,FP32 bytes,INT8 bytes,INT8 SHA-256
0,35,31,112,14527,60466,39315,c5d995fb96809220013197246177ef8be9ad183ba0aab7...


## 4. One-time held-out evaluation

Only now are the Option B test features opened. The threshold and selected checkpoints are frozen. Personal results are at the unique-waveform level; repeated recorder saves cannot multiply the test support. During report reconstruction, this cell reads the saved one-time evaluation and does not re-open the test set.

In [5]:
if REUSE_RUN_DIR:
    RESULTS = json.loads((RUN_DIR / 'final_evaluation.json').read_text(encoding='utf-8'))
    print('Loaded the saved final evaluation; held-out Option B data were not evaluated again.')
else:
    RESULTS = evaluate_all(DATA, WAKE_RUNS, INTENT_RUN)
print('Wake comparison:')
display(pd.DataFrame([
    {'variant': name, 'threshold': r['validation_selected_int8_threshold'],
     'FP32 wake hits/support': f"{r['fp32_heldout_personal_wake_hits']}/{r['fp32_heldout_personal_wake_support']}",
     'FP32 wake recall': r['fp32_heldout_personal_wake_recall'],
     'personal wake recall': r['heldout_personal_wake_recall'],
     'personal wake hits/support': f"{r['heldout_personal_wake_hits']}/{r['heldout_personal_wake_support']}",
     'personal non-wake false accepts': f"{r['false_wake_on_personal_nonwake_clips']}/{r['personal_nonwake_support']}",
     'Option B command false accepts': f"{r['false_wake_on_optionb_command_clips']}/{r['optionb_command_support']}",
     'model bytes': r['int8_model_size_bytes'], 'PC CPU p95 ms': r['pc_cpu_latency']['p95_ms_pc_cpu_model_only']}
    for name, r in RESULTS['wake_model_comparison'].items()
]))
print('Intent comparison:')
display(pd.DataFrame([
    {'model': name,
     'Option B INT8/test accuracy': vals['optionb_test']['accuracy'],
     'Option B macro-F1': vals['optionb_test']['macro_f1_supported_classes'],
     'personal heldout count': vals['personal_heldout_test'].get('count', 0),
     'personal heldout accuracy': vals['personal_heldout_test'].get('accuracy'),
     'personal macro-F1': vals['personal_heldout_test'].get('macro_f1_supported_classes'),
     'size bytes': vals['model_size_bytes'], 'PC CPU p95 ms': vals['pc_cpu_latency']['p95_ms_pc_cpu_model_only']}
    for name, vals in RESULTS['intent_model_comparison'].items()
]))
print('All 31 intent classes on the held-out Option B speakers (canonical vs personalized INT8):')
CLASSIC = RESULTS['intent_model_comparison']['canonical_int8']['optionb_test']['classification_report']
PERSONAL = RESULTS['intent_model_comparison']['personalized_int8']['optionb_test']['classification_report']
display(pd.DataFrame([
    {'class': label, 'support': int(CLASSIC[label]['support']),
     'canonical recall': CLASSIC[label]['recall'], 'personalized recall': PERSONAL[label]['recall'],
     'canonical F1': CLASSIC[label]['f1-score'], 'personalized F1': PERSONAL[label]['f1-score']}
    for label in CLASSIC if label in PERSONAL and isinstance(CLASSIC[label], dict) and 'f1-score' in CLASSIC[label]
]))
print('Your held-out recordings, grouped by recorder label (unmapped labels are prediction-only):')
display(pd.DataFrame([
    {'model': model_name, 'recorded label': source_label,
     'held-out clips': row['heldout_clip_support'], 'mapped intent': row['mapped_intent'],
     'correct': row['correct_hits'], 'accuracy': row['accuracy'],
     'predicted classes': row['predicted_class_counts']}
    for model_name in ('canonical_int8', 'personalized_int8')
    for source_label, row in RESULTS['intent_model_comparison'][model_name]['personal_recorded_commands_all_heldout']['per_recorded_label'].items()
]))
print('Full report:', RUN_DIR / 'final_evaluation.json')
print(json.dumps(RESULTS['known_limitations'], indent=2))


Loaded the saved final evaluation; held-out Option B data were not evaluated again.
Wake comparison:


,variant,threshold,FP32 wake hits/support,FP32 wake recall,personal wake recall,personal wake hits/support,personal non-wake false accepts,Option B command false accepts,model bytes,PC CPU p95 ms
0,broad,0.999867,4/8,0.50,0.500,4/8,0/57,0/1798,37425,0.581605
1,hard,0.998543,6/8,0.75,0.875,7/8,0/57,0/1798,37425,0.543630


Intent comparison:


,model,Option B INT8/test accuracy,Option B macro-F1,personal heldout count,personal heldout accuracy,personal macro-F1,size bytes,PC CPU p95 ms
0,canonical_fp32,0.961068,0.960818,42,0.452381,0.399048,60466,0.42952
1,canonical_int8,0.962736,0.962536,42,0.452381,0.399048,39315,0.50025
2,personalized_fp32,0.956062,0.956133,42,0.880952,0.925556,60466,0.44047
3,personalized_int8,0.959956,0.960235,42,0.880952,0.918148,39315,0.51556


All 31 intent classes on the held-out Option B speakers (canonical vs personalized INT8):


,class,support,canonical recall,personalized recall,canonical F1,personalized F1
0,ALARM_6_00AM,58,0.982759,1.000000,0.974359,1.000000
1,ALARM_8_00AM,56,0.946429,1.000000,0.963636,1.000000
2,ALARM_9_00PM,58,1.000000,1.000000,0.974790,0.983051
3,BRIGHTNESS_100,60,0.983333,0.966667,0.951613,0.906250
4,BRIGHTNESS_20,60,0.983333,0.900000,0.975207,0.931034
5,BRIGHTNESS_60,60,1.000000,1.000000,1.000000,0.991736
6,CALL,54,0.944444,0.962963,0.935780,0.971963
7,COLOR_BLUE,60,0.966667,0.950000,0.950820,0.950000
8,COLOR_GREEN,60,0.933333,0.966667,0.965517,0.974790
9,COLOR_RED,60,1.000000,1.000000,1.000000,1.000000


Your held-out recordings, grouped by recorder label (unmapped labels are prediction-only):


,model,recorded label,held-out clips,mapped intent,correct,accuracy,predicted classes
0,canonical_int8,ALARM_6_00AM,2,ALARM_6_00AM,1.0,0.500000,"{'ALARM_6_00AM': 1, 'LIGHT_OFF': 1}"
1,canonical_int8,ALARM_8_00AM,2,ALARM_8_00AM,0.0,0.000000,{'LIGHT_OFF': 2}
2,canonical_int8,LIGHT_ON,2,LIGHT_ON,1.0,0.500000,"{'VOLUME_UP': 1, 'LIGHT_ON': 1}"
3,canonical_int8,PAUSE,3,PAUSE,0.0,0.000000,"{'PLAY_MUSIC': 2, 'LIST_REMINDERS': 1}"
4,canonical_int8,PLAY_MUSIC,3,PLAY_MUSIC,3.0,1.000000,{'PLAY_MUSIC': 3}
5,canonical_int8,STOP,2,STOP,2.0,1.000000,{'STOP': 2}
6,canonical_int8,alarm_set,3,NaN,NaN,NaN,{'ALARM_8_00AM': 3}
7,canonical_int8,call_mom,2,CALL,0.0,0.000000,"{'LIGHT_ON': 1, 'VOLUME_UP': 1}"
8,canonical_int8,lights_off,5,LIGHT_OFF,0.0,0.000000,"{'ALARM_8_00AM': 1, 'LIGHT_ON': 4}"
9,canonical_int8,lights_on,2,LIGHT_ON,2.0,1.000000,{'LIGHT_ON': 2}


Full report: C:\Users\danda\Desktop\MEng AI Notebooks\AI 222 231\AI 231\Dumalaog_ME2 - Tiny Voice Command Model\runs\personalized-vcm-retrain-20260928-190911\final_evaluation.json
[
  "Personal manifest contains 3 speaker IDs (person-01, person-02, person-03); per-label clip splitting permits speaker overlap across train/validation/test, so these results do not establish unseen-speaker accuracy.",
  "Personal held-out support is low for several intent classes and is reported per class.",
  "Option B test clips have prior project evaluations and are not a pristine first-use benchmark.",
  "Command-window false wake counts are not continuous-audio false activations per hour; no Pi hardware was used.",
  "INT8 latency here is local Windows PC CPU model-only latency, not Raspberry Pi frontend-plus-model latency.",
  "This experiment does not automatically replace the canonical model files or change the Pi release."
]


## Interpretation and promotion gate

Choose the wake variant using the held-out personal wake and non-wake results, with the speaker-overlap limitation in mind. Keep the simpler broad-negative version if personal hard negatives do not improve wake recall without added false accepts. Keep the canonical intent model if the personalized model's personal gain damages the Option B speaker-disjoint score. These are local research candidates; exporting ONNX does not promote them to the UI or Raspberry Pi. Collect more speakers, sessions, wake near-misses, and room noise before treating them as general-purpose.